<img src='./Images/idealized_flood_guard.jpeg' style='width: 100%; height: 400px; object-fit: cover;' />

---
<h1 align='center'>
NAIROBI FLOOD GUARD
</h1>

> **Authors**: Group 4

---

<h2 align='center'>
1. INTRODUCTION
</h2>

### Overview

Nairobi Flood Guard is a data science project that addresses the growing threat of flodding across Kenya. Motivated by the devastating April 2024 floods and the recent 2026 floods that disrupted lives across the country, the project aims to shift flood response from reactive to predictive.

The project has two components:

- The first is a **flood susceptibility model** that combines NASA SRTM terrain data and CHIRPS satellite rainfall data to predict which of Kenya's 1,450 administrative ward are most at risk of flooding, trained on UNOSAT satellite-derived flood extent data from the April 2024 event.

- The second is a **matatu route optimization system** that overlays floods risk predictions against Nairobi's public transport network to identify affected routes and recommend safer alternatives for operators and commuters.

Built entirely on open data and reproducible tools, Nairobi Flood Guard gives communities, emergency responders, and transport operations the advance information they need to act before a flood, not after.

### Business Understanding

#### *Problem Statement*

Flooding in Nairobi and Kenya at large causes loss of life, displacement and infrastructure damage. The April 2024 floods killed over 200 people and displaced hundreds of thousands. Current flood response is largely reactive rather than predictive. There is no system that warns communities or road users in advance.

#### *Objectives*

- **Flood Susceptibility Prediction** - identify which wards are at highest risk of flooding given terrain and rainfall conditions

- **Matatu Route Optimization** - given predicted flood zones, identify affected matatu routes and recommend safer alternatives

#### *Stakeholders*

- **Kenya Red Cross / National Disaster Management Unit** - early warning for evacuation planning

- **Nairobi City County** - infrastructure and emergency response

- **Matatu operators and SACCOs** - route planning during flood events

- **Commuters** - real-time route guidance

- **General public in flood-prone wards** - advance warning to evacuate or prepare

#### *Success Metrics*

- High **recall** on the flood prediction model - missing a flood (false negative) is far more costly than a false alarm

- Route recommendations that successfully avoid confirmed flood zones from UNOSAT data

- Ward-level risk scores that align with known historically flooded areas

#### *Scope and Limitations*

- Labels are based on a **single flood event** (April 2024) - the model may not generalise to floods caused by different conditions

- GTFS data is from **2019** - the matatu network may have changed

- Ward-level predictions are coarse - a ward may be partially flooded but the whole ward gets labelled as flooded

- The model predicts **susceptibiity** not exact flood timing or depth

---

<h2 align='center'>
2. DATA UNDERSTANDING
</h2>

This project utilises five datasets, each contributing a different dimension to the flood prediction and route optimization pipeline. Below is an overview of each dataset, followed by an exploratory examination of the compiled feature matrix

### a) SRTM Digital Elevation Model (DEM)

The Shuttle Radar Topography Mission (SRTM) DEM, acquired by NASA in 2000, provides elevation data at 90 metre resolution. It was used to derive three terrain features per ward: mean elevation, minimum elevation, maximum elevation, and slope. These features capture how water naturally flows and accumulates across the landscape.

**Source:** OpenTopography (SRTM GL3 product)

### b) CHIRPS Rainfall Data

The Climate Hazards Group InfraRed Precipitation with Station data (CHIRPS) provides daily rainfall estimates at approximately 5km resolution. Ninety daily rasters covering February–April 2024 were used to derive three rainfall features per ward: cumulative rainfall, maximum single-day rainfall, and total rainfall in the seven days preceding the April 26 flood event.

**Source:** UCSB Climate Hazards Group

### c) UNOSAT Flood Extent - FL20240426KEN

A satellite-derived flood extent geodatabase produced by UNOSAT following the April 2024 Kenya floods. The Kenya-wide maximum flood water extent polygon was used to generate binary flood labels for each ward — flooded (1) or not flooded (0).

**Source:** UNOSAT / UNITAR

### d) Kenya Wards Shapefile

A polygon shapefile of Kenya's 1450 administrative wards including ward name, sub-county, county, and 2009 census population. This served as the spatial backbone of the project - all raster datasets were aggregated to ward level through spatial joins and zonal statistics.

**Source:** Regional Centre for Mapping of Resources for Development (RGMRD)

### e) GTFS Feed 2019 - Nairobi Matatu Network

A General Transit Feed Specification (GTFS) dataset describing Nairobi's matatu public transport network as of 2019, including 136 routes, 4,284 stops, and 36,483 route shape points. This dataset underpins the route optimization component of the project.

**Source:** Digital Matatus Project

### f) Compiled Feature Matrix - floods.gpkg

All datasets were processed and merged into a single GeoPackage file (`floods.gpkg`) containing one row per ward with all features and the flood label. The cells below examine this compiled dataset.

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = gpd.read_file('./Data/floods.gpkg')
df.head()

,ward,subcounty,county,pop2009,flooded,rain_cumulative_mm,rain_max_daily_mm,rain_preflood_7d_mm,elevation_mean_m,elevation_min_m,elevation_max_m,slope_mean_deg,geometry
0,WABERA,Isiolo Sub County,ISIOLO,17431.0,0,0.000000,0.000000,0.000000,1099.219448,1026.0,1177.0,1.413028,"MULTIPOLYGON (((37.59968 0.40029, 37.59976 0.4..."
1,North Kamagambo Ward,Rongo Sub County,Migori,18755.0,0,258.608840,14.135786,39.955728,1405.549444,1350.0,1508.0,2.246821,"MULTIPOLYGON (((34.59938 -0.65054, 34.60006 -0..."
2,Central Kamagambo Ward,Rongo Sub County,Migori,27756.0,0,125.951383,8.779010,20.537484,1462.532567,1387.0,1534.0,3.392240,"MULTIPOLYGON (((34.61175 -0.73357, 34.61183 -0..."
3,South Kamagambo Ward,Rongo Sub County,Migori,27179.0,0,45.941367,5.696220,10.762588,1490.546337,1363.0,1638.0,4.263301,"MULTIPOLYGON (((34.61751 -0.87293, 34.6175 -0...."
4,North Sakwa Ward,Awendo Sub County,Migori,22874.0,0,78.351185,4.580237,10.795914,1376.935386,1287.0,1617.0,3.834336,"MULTIPOLYGON (((34.55349 -0.75193, 34.55364 -0..."
